# MDP Problem Solving Assignment

## Applied Machine Intelligence

This notebook solves a set of Markov Decision Process (MDP) problems involving:

- Bellman expectation updates
- Policy evaluation
- Value iteration
- Optimal policy selection
- Conceptual reasoning about MDPs

Unless otherwise stated, the discount factor used is:

\[
\gamma = 0.9
\]

The MDP contains three states:

- \(S_0\)
- \(S_1\)
- \(S_2\), which is a terminal state

In [1]:
import pandas as pd

# Discount factor
gamma = 0.9

# States and actions
states = ["S0", "S1", "S2"]
actions = ["A0", "A1"]

# Transition and reward model
# Format: (current_state, action): (next_state, reward)
transitions = {
    ("S0", "A0"): ("S1", 5),
    ("S0", "A1"): ("S2", 2),
    ("S1", "A0"): ("S2", 4),
    ("S1", "A1"): ("S0", 1),
    ("S2", "A0"): ("S2", 0),
    ("S2", "A1"): ("S2", 0),
}

# Fixed policy
policy = {
    "S0": "A0",
    "S1": "A1",
    "S2": "A0"
}

print("Discount factor:", gamma)
print("States:", states)
print("Fixed policy:", policy)

Discount factor: 0.9
States: ['S0', 'S1', 'S2']
Fixed policy: {'S0': 'A0', 'S1': 'A1', 'S2': 'A0'}


## Problem 1: One-Step Bellman Expectation Backup

The Bellman expectation equation under a fixed policy is:

\[
V_{k+1}^{\pi}(s)
=
\sum_a \pi(a|s)
\sum_{s'} P(s'|s,a)
\left[
R(s,a,s')+\gamma V_k^{\pi}(s')
\right]
\]

Because the policy is deterministic, only the action selected by the policy needs to be evaluated.

The initial state values are:

\[
V_0(S_0)=V_0(S_1)=V_0(S_2)=0
\]

### State \(S_0\)

The policy selects \(A_0\), which transitions to \(S_1\) with reward 5:

\[
V_1(S_0)
=
5+0.9V_0(S_1)
=
5+0.9(0)
=
5
\]

### State \(S_1\)

The policy selects \(A_1\), which transitions to \(S_0\) with reward 1:

\[
V_1(S_1)
=
1+0.9V_0(S_0)
=
1+0.9(0)
=
1
\]

### State \(S_2\)

State \(S_2\) is terminal and remains in \(S_2\) with reward 0:

\[
V_1(S_2)
=
0+0.9V_0(S_2)
=
0
\]

In [2]:
# Initial state values
V0 = {
    "S0": 0.0,
    "S1": 0.0,
    "S2": 0.0
}

# Perform one Bellman expectation backup
V1 = {}

for state in states:
    action = policy[state]
    next_state, reward = transitions[(state, action)]
    
    V1[state] = reward + gamma * V0[next_state]

# Create a results table
problem1_results = pd.DataFrame({
    "State": states,
    "Policy Action": [policy[state] for state in states],
    "Initial Value V0": [V0[state] for state in states],
    "Updated Value V1": [V1[state] for state in states]
})

problem1_results

,State,Policy Action,Initial Value V0,Updated Value V1
0,S0,A0,0.0,5.0
1,S1,A1,0.0,1.0
2,S2,A0,0.0,0.0


### Final Answer — Problem 1

After the first Bellman expectation update:

\[
\boxed{V_1(S_0)=5}
\]

\[
\boxed{V_1(S_1)=1}
\]

\[
\boxed{V_1(S_2)=0}
\]

The updated values represent the immediate rewards received from following the fixed policy because all initial future-state values were zero.

## Problem 2: Policy Evaluation

Policy evaluation repeatedly applies the Bellman expectation equation until the state values converge:

\[
V^\pi(s)
=
R(s,\pi(s),s')
+
\gamma V^\pi(s')
\]

Under the fixed policy:

- In \(S_0\), action \(A_0\) moves to \(S_1\) with reward 5.
- In \(S_1\), action \(A_1\) moves to \(S_0\) with reward 1.
- \(S_2\) is terminal and has value 0.

Therefore, the Bellman equations are:

\[
V^\pi(S_0)=5+0.9V^\pi(S_1)
\]

\[
V^\pi(S_1)=1+0.9V^\pi(S_0)
\]

\[
V^\pi(S_2)=0
\]

Because the policy repeatedly moves between \(S_0\) and \(S_1\), the agent continues receiving rewards 5 and 1. The discount factor ensures that the infinite sequence of future rewards has a finite value.

In [3]:
# Initialize all state values to zero
V_policy = {
    "S0": 0.0,
    "S1": 0.0,
    "S2": 0.0
}

# Convergence settings
tolerance = 1e-10
max_iterations = 10_000

# Store selected iterations for inspection
evaluation_history = []

for iteration in range(1, max_iterations + 1):
    
    # Synchronous Bellman expectation updates
    new_values = {
        "S0": 5 + gamma * V_policy["S1"],
        "S1": 1 + gamma * V_policy["S0"],
        "S2": 0.0
    }
    
    # Calculate the maximum change across all states
    delta = max(
        abs(new_values[state] - V_policy[state])
        for state in states
    )
    
    # Save early iterations and the final iteration
    if iteration <= 10:
        evaluation_history.append({
            "Iteration": iteration,
            "V(S0)": new_values["S0"],
            "V(S1)": new_values["S1"],
            "V(S2)": new_values["S2"],
            "Maximum Change": delta
        })
    
    # Replace the old values with the updated values
    V_policy = new_values
    
    # Stop when the values have converged
    if delta < tolerance:
        evaluation_history.append({
            "Iteration": iteration,
            "V(S0)": V_policy["S0"],
            "V(S1)": V_policy["S1"],
            "V(S2)": V_policy["S2"],
            "Maximum Change": delta
        })
        break

print(f"Policy evaluation converged after {iteration} iterations.")
print(f"Final maximum change: {delta:.12f}")

Policy evaluation converged after 235 iterations.
Final maximum change: 0.000000000098


In [4]:
problem2_results = pd.DataFrame({
    "State": states,
    "Policy Action": [policy[state] for state in states],
    "Converged Value": [V_policy[state] for state in states]
})

problem2_results["Converged Value"] = (
    problem2_results["Converged Value"].round(6)
)

problem2_results

,State,Policy Action,Converged Value
0,S0,A0,31.052632
1,S1,A1,28.947368
2,S2,A0,0.000000


### Algebraic Validation

The converged values can also be verified by solving the Bellman equations directly.

Starting with:

\[
V^\pi(S_0)=5+0.9V^\pi(S_1)
\]

\[
V^\pi(S_1)=1+0.9V^\pi(S_0)
\]

Substitute the second equation into the first:

\[
V^\pi(S_0)
=
5+0.9\left(1+0.9V^\pi(S_0)\right)
\]

Expand the equation:

\[
V^\pi(S_0)
=
5+0.9+0.81V^\pi(S_0)
\]

\[
V^\pi(S_0)
=
5.9+0.81V^\pi(S_0)
\]

Move the value term to the left:

\[
V^\pi(S_0)-0.81V^\pi(S_0)=5.9
\]

\[
0.19V^\pi(S_0)=5.9
\]

Therefore:

\[
V^\pi(S_0)
=
\frac{5.9}{0.19}
=
31.0526
\]

Now substitute this value into the equation for \(S_1\):

\[
V^\pi(S_1)
=
1+0.9(31.0526)
\]

\[
V^\pi(S_1)
=
28.9474
\]

The terminal-state value remains:

\[
V^\pi(S_2)=0
\]

### Final Answer — Problem 2

After repeated Bellman expectation updates, the policy values converge to:

\[
\boxed{V^\pi(S_0)=31.0526}
\]

\[
\boxed{V^\pi(S_1)=28.9474}
\]

\[
\boxed{V^\pi(S_2)=0}
\]

The values of \(S_0\) and \(S_1\) are relatively large because the fixed policy creates a continuing cycle:

\[
S_0 \rightarrow S_1 \rightarrow S_0
\]

During this cycle, the agent repeatedly receives rewards of 5 and 1. Future rewards are discounted by \(\gamma=0.9\), which keeps the total return finite.

## Problem 3: One Iteration of Value Iteration

The Bellman optimality backup is:

\[
V_{k+1}(s)
=
\max_a
\sum_{s'}P(s'|s,a)
\left[
R(s,a,s')+\gamma V_k(s')
\right]
\]

Because the transition model is deterministic, the equation simplifies to:

\[
V_{k+1}(s)
=
\max_a
\left[
R(s,a,s')+\gamma V_k(s')
\right]
\]

The initial values are:

\[
V_0(S_0)=V_0(S_1)=V_0(S_2)=0
\]

### State \(S_0\)

For action \(A_0\):

\[
Q_0(S_0,A_0)
=
5+0.9V_0(S_1)
=
5
\]

For action \(A_1\):

\[
Q_0(S_0,A_1)
=
2+0.9V_0(S_2)
=
2
\]

Therefore:

\[
V_1(S_0)=\max(5,2)=5
\]

### State \(S_1\)

For action \(A_0\):

\[
Q_0(S_1,A_0)
=
4+0.9V_0(S_2)
=
4
\]

For action \(A_1\):

\[
Q_0(S_1,A_1)
=
1+0.9V_0(S_0)
=
1
\]

Therefore:

\[
V_1(S_1)=\max(4,1)=4
\]

### State \(S_2\)

State \(S_2\) is terminal:

\[
V_1(S_2)=0
\]

In [5]:
# Initialize the state values
V0_value_iteration = {
    "S0": 0.0,
    "S1": 0.0,
    "S2": 0.0
}

q_values_iteration1 = {}
V1_value_iteration = {}

for state in states:
    
    if state == "S2":
        q_values_iteration1[(state, "A0")] = 0.0
        q_values_iteration1[(state, "A1")] = 0.0
        V1_value_iteration[state] = 0.0
        continue
    
    action_values = {}
    
    for action in actions:
        next_state, reward = transitions[(state, action)]
        
        action_values[action] = (
            reward
            + gamma * V0_value_iteration[next_state]
        )
        
        q_values_iteration1[(state, action)] = action_values[action]
    
    # Bellman optimality backup
    V1_value_iteration[state] = max(action_values.values())

problem3_results = pd.DataFrame({
    "State": states,
    "Q(S, A0)": [
        q_values_iteration1[(state, "A0")]
        for state in states
    ],
    "Q(S, A1)": [
        q_values_iteration1[(state, "A1")]
        for state in states
    ],
    "Updated Value V1": [
        V1_value_iteration[state]
        for state in states
    ]
})

problem3_results

,State,"Q(S, A0)","Q(S, A1)",Updated Value V1
0,S0,5.0,2.0,5.0
1,S1,4.0,1.0,4.0
2,S2,0.0,0.0,0.0


### Final Answer — Problem 3

After one full Bellman optimality backup:

\[
\boxed{V_1(S_0)=5}
\]

\[
\boxed{V_1(S_1)=4}
\]

\[
\boxed{V_1(S_2)=0}
\]

For \(S_0\), action \(A_0\) produces the larger one-step return.

For \(S_1\), action \(A_0\) produces the larger one-step return.

Because all initial state values are zero, the first value-iteration update is determined entirely by the immediate rewards.

## Problem 4: Optimal Policy

The optimal policy selects the action with the largest optimal Q-value in each state.

The Bellman optimality equation is:

\[
V^*(s)
=
\max_a
\left[
R(s,a,s')+\gamma V^*(s')
\right]
\]

The optimal action-value function is:

\[
Q^*(s,a)
=
R(s,a,s')+\gamma V^*(s')
\]

Value iteration will be repeated until the state values converge. The final Q-values will then be calculated and compared to determine the greedy optimal policy.

In [6]:
# Initialize optimal state values
V_optimal = {
    "S0": 0.0,
    "S1": 0.0,
    "S2": 0.0
}

tolerance = 1e-10
max_iterations = 10_000

for iteration in range(1, max_iterations + 1):
    
    new_values = {}
    
    for state in states:
        
        if state == "S2":
            new_values[state] = 0.0
            continue
        
        action_values = []
        
        for action in actions:
            next_state, reward = transitions[(state, action)]
            
            q_value = reward + gamma * V_optimal[next_state]
            action_values.append(q_value)
        
        new_values[state] = max(action_values)
    
    delta = max(
        abs(new_values[state] - V_optimal[state])
        for state in states
    )
    
    V_optimal = new_values
    
    if delta < tolerance:
        break

print(f"Value iteration converged after {iteration} iterations.")
print(f"Final maximum change: {delta:.12f}")
print("Optimal state values:", V_optimal)

Value iteration converged after 233 iterations.
Final maximum change: 0.000000000097
Optimal state values: {'S0': 31.05263157831625, 'S1': 28.94736842044829, 'S2': 0.0}


In [7]:
optimal_q_values = {}
optimal_policy = {}

for state in states:
    
    if state == "S2":
        optimal_q_values[(state, "A0")] = 0.0
        optimal_q_values[(state, "A1")] = 0.0
        optimal_policy[state] = "Terminal"
        continue
    
    state_action_values = {}
    
    for action in actions:
        next_state, reward = transitions[(state, action)]
        
        q_value = reward + gamma * V_optimal[next_state]
        
        optimal_q_values[(state, action)] = q_value
        state_action_values[action] = q_value
    
    optimal_policy[state] = max(
        state_action_values,
        key=state_action_values.get
    )

problem4_results = pd.DataFrame({
    "State": states,
    "Q*(S, A0)": [
        optimal_q_values[(state, "A0")]
        for state in states
    ],
    "Q*(S, A1)": [
        optimal_q_values[(state, "A1")]
        for state in states
    ],
    "Greedy Optimal Action": [
        optimal_policy[state]
        for state in states
    ]
})

problem4_results[["Q*(S, A0)", "Q*(S, A1)"]] = (
    problem4_results[["Q*(S, A0)", "Q*(S, A1)"]].round(6)
)

problem4_results

,State,"Q*(S, A0)","Q*(S, A1)",Greedy Optimal Action
0,S0,31.052632,2.000000,A0
1,S1,4.000000,28.947368,A1
2,S2,0.000000,0.000000,Terminal


### Q-Value Comparison

For state \(S_0\):

\[
Q^*(S_0,A_0)
=
5+0.9V^*(S_1)
\]

\[
Q^*(S_0,A_0)
=
5+0.9(28.9474)
=
31.0526
\]

\[
Q^*(S_0,A_1)
=
2+0.9V^*(S_2)
=
2
\]

Therefore, the optimal action is:

\[
\boxed{\pi^*(S_0)=A_0}
\]

For state \(S_1\):

\[
Q^*(S_1,A_0)
=
4+0.9V^*(S_2)
=
4
\]

\[
Q^*(S_1,A_1)
=
1+0.9V^*(S_0)
\]

\[
Q^*(S_1,A_1)
=
1+0.9(31.0526)
=
28.9474
\]

Therefore, the optimal action is:

\[
\boxed{\pi^*(S_1)=A_1}
\]

### Final Answer — Problem 4

The optimal policy is:

\[
\boxed{\pi^*(S_0)=A_0}
\]

\[
\boxed{\pi^*(S_1)=A_1}
\]

State \(S_2\) is terminal.

Although \(A_0\) in \(S_1\) gives a larger immediate reward, \(A_1\) is optimal because it returns the agent to \(S_0\), allowing it to continue earning discounted rewards through the \(S_0\)-\(S_1\) cycle.

## Problem 5: Concept Check

### 1. Why can a poorly designed reward function lead to unintended agent behavior?

A reinforcement learning agent tries to maximize the reward defined by the designer, not necessarily the designer's true intention. If the reward function is incomplete or poorly specified, the agent may discover shortcuts that produce high rewards without achieving the desired outcome. This behavior is often called **reward hacking** or **specification gaming**. Therefore, reward functions must represent both the main objective and important constraints.

### 2. Why does the Bellman optimality equation use a maximum over actions, while the Bellman expectation equation uses an average over actions?

The Bellman expectation equation evaluates a particular policy, so it combines action values according to the probabilities assigned by that policy. If the policy is stochastic, this produces a probability-weighted average over actions. The Bellman optimality equation instead searches for the best possible behavior, so it selects the action with the highest expected return. Therefore, expectation is used for policy evaluation, while maximization is used for finding the optimal policy.

### 3. What makes terminal states easier to handle in dynamic programming methods?

A terminal state ends the decision-making process and normally has no future rewards. Its value is typically fixed at zero, so it does not require repeated updates during policy evaluation or value iteration. This provides a clear boundary condition for Bellman equations. As a result, terminal states simplify both the calculations and convergence process.

## Problem 6: Extension

The transition for \(S_1\) with action \(A_0\) is changed from:

\[
S_1 \xrightarrow{A_0,\;r=4} S_2
\]

to:

\[
S_1 \xrightarrow{A_0,\;r=3} S_0
\]

### Effect on the Value of \(S_1\)

The value of \(S_1\) would increase because action \(A_0\) no longer sends the agent to the terminal state. Although its immediate reward decreases from 4 to 3, the agent now returns to \(S_0\), where it can continue receiving future rewards. Therefore, the discounted long-term return from \(S_1\) becomes more important than the smaller change in immediate reward.

### Effect on the Optimal Policy in \(S_1\)

Under the modified transition, both actions available in \(S_1\) lead to \(S_0\):

- \(A_0\) leads to \(S_0\) with reward 3.
- \(A_1\) leads to \(S_0\) with reward 1.

Because both actions have the same next state, they receive the same discounted future-state value. Action \(A_0\) provides the larger immediate reward, so it becomes the optimal action in \(S_1\):

\[
\boxed{\pi^*(S_1)=A_0}
\]

### Effect on Cyclic Behavior

The modified MDP becomes more cyclic because action \(A_0\) in \(S_1\) now returns the agent to \(S_0\) instead of terminating the process. Combined with action \(A_0\) in \(S_0\), which moves the agent to \(S_1\), this produces the continuing cycle:

\[
S_0 \xrightarrow{A_0} S_1
\xrightarrow{A_0} S_0
\]

The agent can repeatedly move between \(S_0\) and \(S_1\) while collecting rewards. The terminal state \(S_2\) becomes less likely to be reached under the optimal policy.

### Final Answer — Problem 6

The transition change increases the long-term value of \(S_1\), changes its optimal action from \(A_1\) to \(A_0\), and makes the MDP more cyclic. The agent now has a stronger incentive to remain in the \(S_0\)-\(S_1\) loop instead of moving to the terminal state.

In [8]:
# Expected analytical results
expected_policy_values = {
    "S0": 31.0526315789,
    "S1": 28.9473684211,
    "S2": 0.0
}

expected_first_value_iteration = {
    "S0": 5.0,
    "S1": 4.0,
    "S2": 0.0
}

expected_optimal_policy = {
    "S0": "A0",
    "S1": "A1",
    "S2": "Terminal"
}

# Validate Problem 2
policy_value_checks = {
    state: abs(V_policy[state] - expected_policy_values[state]) < 1e-6
    for state in states
}

# Validate Problem 3
value_iteration_checks = {
    state: abs(
        V1_value_iteration[state]
        - expected_first_value_iteration[state]
    ) < 1e-6
    for state in states
}

# Validate Problem 4
optimal_policy_checks = {
    state: optimal_policy[state] == expected_optimal_policy[state]
    for state in states
}

validation_results = pd.DataFrame({
    "Problem": [
        "Problem 2: Policy Evaluation",
        "Problem 3: One Value-Iteration Backup",
        "Problem 4: Optimal Policy"
    ],
    "Validation Passed": [
        all(policy_value_checks.values()),
        all(value_iteration_checks.values()),
        all(optimal_policy_checks.values())
    ]
})

validation_results

,Problem,Validation Passed
0,Problem 2: Policy Evaluation,True
1,Problem 3: One Value-Iteration Backup,True
2,Problem 4: Optimal Policy,True


## Assignment Summary

This notebook solved six Markov Decision Process problems using a discount factor of \(\gamma=0.9\).

The main results were:

- The first Bellman expectation update produced values of 5, 1, and 0 for \(S_0\), \(S_1\), and \(S_2\).
- Iterative policy evaluation converged to \(V^\pi(S_0)=31.0526\), \(V^\pi(S_1)=28.9474\), and \(V^\pi(S_2)=0\).
- The first Bellman optimality backup produced values of 5, 4, and 0.
- The optimal actions were \(A_0\) in \(S_0\) and \(A_1\) in \(S_1\).
- The conceptual questions demonstrated how reward design, policy expectations, maximization, and terminal states affect MDP solutions.
- Modifying the transition for \(S_1,A_0\) made the system more cyclic and changed the optimal action in \(S_1\).

All numerical results were validated programmatically.